# LoomEncoder Demo

**Field-level tokenization**: every field of every structured instance becomes
its own token. `LoomCompiler.build_encoder()` creates a `LoomEncoder` that
collates raw Python data into columnar tensors, embeds them via per-branch
modules, and produces `[B, N_max, d_model]` output ready for any transformer.

Full round-trip:
`raw dicts → encoder.collate() → encoder() → TransformerEncoder → LoomHead.decode()`

## 1. Setup

In [ ]:
import torch
import torch.nn as nn

from loomlib import (
    LoomModel, LoomUnion, LoomCompiler,
    Boolean, Scalar, ContinuousScalar,
)

class Steering(LoomModel):
    angle: ContinuousScalar[-1.0, 1.0]
    throttle: ContinuousScalar[0.0, 1.0]

class Shooting(LoomModel):
    active: Boolean()
    power: ContinuousScalar[0.0, 10.0]

class GameAction(LoomUnion):
    steer: Steering
    shoot: Shooting

D_MODEL = 64

encoder = LoomCompiler.build_encoder(GameAction, d_model=D_MODEL)
head = LoomCompiler.build_head(GameAction, d_model=D_MODEL)

print(f"Branches: {encoder.branch_names}")
for bname, fields in encoder.branch_fields.items():
    print(f"  {bname}: { {k: repr(v) for k, v in fields.items()} }")
print(f"Total unique fields: {encoder._total_fields}")
print(f"\n{head.allocation.pretty_print()}")

Branches: ['steer', 'shoot']
  steer: {'angle': 'ContinuousScalar(-1.0, 1.0)', 'throttle': 'ContinuousScalar(0.0, 1.0)'}
  shoot: {'active': 'Boolean()', 'power': 'ContinuousScalar(0.0, 10.0)'}
Total unique fields: 4

SliceAllocation:
  z[0:2]  __opcode__                      Categorical(2)
  z[2:3]  steer.angle                     ContinuousScalar(-1.0, 1.0)
  z[3:4]  steer.throttle                  ContinuousScalar(0.0, 1.0)
  z[4:5]  shoot.active                    Boolean()
  z[5:6]  shoot.power                     ContinuousScalar(0.0, 10.0)
  Total logits: 6


## 2. Collation

`encoder.collate()` converts raw instance sequences into a `LoomBatch` --
five columnar tensors: `type_ids`, `inst_ids`, `field_ids`, `values`, `padding_mask`.

In [2]:
batch_data = [
    # sequence 0: three instances
    [("steer", {"angle": 0.5, "throttle": 0.8}),
     ("shoot", {"active": True, "power": 3.0}),
     ("steer", {"angle": -0.2, "throttle": 0.1})],
    # sequence 1: one instance (shorter -> padded)
    [("shoot", {"active": False, "power": -1.0})],
]

batch = encoder.collate(batch_data)

print(f"Batch: {batch.batch_size} sequences x {batch.seq_len} tokens (max)")
for name in ["type_ids", "inst_ids", "field_ids", "values", "padding_mask"]:
    t = getattr(batch, name)
    print(f"  {name:13s} {str(t.dtype):12s} {list(t.shape)}")

for b in range(batch.batch_size):
    n_real = (~batch.padding_mask[b]).sum().item()
    n_pad = batch.padding_mask[b].sum().item()
    print(f"\nSequence {b} ({n_real} real, {n_pad} padding):")
    print(f"  type_ids:     {batch.type_ids[b].tolist()}")
    print(f"  inst_ids:     {batch.inst_ids[b].tolist()}")
    print(f"  field_ids:    {batch.field_ids[b].tolist()}")
    print(f"  values:       {[f'{v:.4f}' for v in batch.values[b].tolist()]}")
    print(f"  padding_mask: {batch.padding_mask[b].int().tolist()}")

# seq 0: steer(2) + shoot(2) + steer(2) = 6; seq 1: shoot(2) padded to 6
assert batch.type_ids.shape == (2, 6)
assert batch.padding_mask[0].sum() == 0
assert batch.padding_mask[1].sum() == 4
print("\n✓ Collation correct")

Batch: 2 sequences x 6 tokens (max)
  type_ids      torch.int64  [2, 6]
  inst_ids      torch.int64  [2, 6]
  field_ids     torch.int64  [2, 6]
  values        torch.float32 [2, 6]
  padding_mask  torch.bool   [2, 6]

Sequence 0 (6 real, 0 padding):
  type_ids:     [0, 0, 1, 1, 0, 0]
  inst_ids:     [0, 0, 1, 1, 2, 2]
  field_ids:    [0, 1, 2, 3, 0, 1]
  values:       ['0.5493', '0.6931', '6.0000', '3.0000', '-0.2027', '-1.0986']
  padding_mask: [0, 0, 0, 0, 0, 0]

Sequence 1 (2 real, 4 padding):
  type_ids:     [1, 1, 0, 0, 0, 0]
  inst_ids:     [0, 0, 0, 0, 0, 0]
  field_ids:    [2, 3, 0, 0, 0, 0]
  values:       ['-6.0000', '-1.0000', '0.0000', '0.0000', '0.0000', '0.0000']
  padding_mask: [0, 0, 1, 1, 1, 1]

✓ Collation correct


## 3. Forward Pass and Round-Trip

`encoder(batch)` dispatches tokens to per-branch embedding modules, adds
shared type/instance positional embeddings, and returns `[B, N_max, d_model]`.
We feed this through a `TransformerEncoder` and back into `LoomHead.decode()`.

In [3]:
emb = encoder(batch)
print(f"Encoder output: {emb.shape}")
print(f"Padding norms (seq 1, pos 2-5): "
      f"{[f'{v:.2f}' for v in emb[1, 2:].norm(dim=-1).tolist()]}  (expect 0.00)")

# Transformer
transformer = nn.TransformerEncoder(
    nn.TransformerEncoderLayer(
        d_model=D_MODEL, nhead=4, dim_feedforward=128, batch_first=True,
    ),
    num_layers=2,
)
hidden = transformer(emb, src_key_padding_mask=batch.padding_mask)
print(f"Transformer output: {hidden.shape}")

# Pool -> decode
real = ~batch.padding_mask
pooled = (hidden * real.unsqueeze(-1).float()).sum(dim=1) / real.sum(dim=1, keepdim=True).float()
z = head(pooled)
decoded = head.decode(z)
print(f"\nDecoded output:")
for key, val in decoded.items():
    print(f"  {key}: {val}")

# Gradient flow
z.sum().backward()
for name, mod in [("encoder", encoder), ("transformer", transformer), ("head", head)]:
    has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in mod.parameters())
    print(f"  {name} gradients: {has_grad}")
    assert has_grad
print("\n✓ End-to-end round-trip with gradient flow verified")

Encoder output: torch.Size([2, 6, 64])
Padding norms (seq 1, pos 2-5): ['0.00', '0.00', '0.00', '0.00']  (expect 0.00)
Transformer output: torch.Size([2, 6, 64])

Decoded output:
  __opcode__: tensor([0, 1])
  steer.angle: tensor([-0.1813, -0.3880], grad_fn=<AddBackward0>)
  steer.throttle: tensor([0.4914, 0.6961], grad_fn=<AddBackward0>)
  shoot.active: tensor([False,  True])
  shoot.power: tensor([-0.0304, -0.2264], grad_fn=<SqueezeBackward1>)


  encoder gradients: True
  transformer gradients: True
  head gradients: True

✓ End-to-end round-trip with gradient flow verified


## 4. Custom Branch Embedding

Each branch uses a `DefaultBranchEmbedding` (additive field identity + scaled value)
by default. Pass a custom `nn.Module` matching
`forward(field_ids: [N], values: [N]) -> [N, d_model]` to override any branch.

In [4]:
class MLPBranchEmbedding(nn.Module):
    """MLP on the scalar value instead of the default additive scheme."""
    def __init__(self, num_fields, d_model):
        super().__init__()
        self.field_embedding = nn.Embedding(num_fields, d_model)
        self.mlp = nn.Sequential(nn.Linear(1, d_model), nn.ReLU(), nn.Linear(d_model, d_model))

    def forward(self, field_ids, values):
        return self.field_embedding(field_ids) + self.mlp(values.unsqueeze(-1))

custom_encoder = LoomCompiler.build_encoder(
    GameAction, d_model=D_MODEL,
    branch_embeddings={"shoot": MLPBranchEmbedding(num_fields=2, d_model=D_MODEL)},
)

print(f"steer: {type(custom_encoder.branch_encoders['steer']).__name__}")
print(f"shoot: {type(custom_encoder.branch_encoders['shoot']).__name__}")

out = custom_encoder(batch)
print(f"Output: {out.shape}")

out.sum().backward()
for bname in custom_encoder.branch_names:
    mod = custom_encoder.branch_encoders[bname]
    has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in mod.parameters())
    print(f"  {bname} branch gradients: {has_grad}")
print("\n✓ Custom per-branch embedding works")

steer: DefaultBranchEmbedding
shoot: MLPBranchEmbedding
Output: torch.Size([2, 6, 64])
  steer branch gradients: True
  shoot branch gradients: True

✓ Custom per-branch embedding works


## Summary

`LoomCompiler.build_encoder()` creates a `LoomEncoder` that:
1. **Collates** raw Python data into a `LoomBatch` of 5 columnar tensors
2. **Embeds** field-tokens via per-branch modules (default: additive field identity + scaled value)
3. **Outputs** `[B, N_max, d_model]` ready for any transformer

The `padding_mask` follows PyTorch convention (True = padding) and plugs
directly into `nn.TransformerEncoder(src_key_padding_mask=...)`.
Per-branch embedding modules are user-replaceable.